In [0]:
%pip install --upgrade databricks-sdk

In [0]:
dbutils.library.restartPython() 

In [0]:
import base64
import os
from databricks.sdk import WorkspaceClient
from PIL import Image 
from config import DeployConfig
from io import BytesIO

In [0]:
endpoint='shovakeemian-gpt-4o'

In [0]:
dbutils.widgets.text("config_path", "./config/env_variables.yml")
config_path = dbutils.widgets.get("config_path")
cfg = DeployConfig.from_yaml(config_path)

In [0]:
image_table = getattr(cfg, f"image_table")
brand_table = getattr(cfg, f"brand_table")

In [0]:
image_path='../streamlit-app/images/bulldog_ad.png'

In [0]:
pet_image=spark.sql(f'select content from {image_table.path} where id=25').collect()[0]['content']
Image.open(BytesIO(pet_image))

In [0]:
image_data = base64.standard_b64encode(pet_image).decode("utf-8")

In [0]:
type(image_data)

In [0]:
#image->text
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
openai_client = w.serving_endpoints.get_open_ai_client()

pet_image=spark.sql(f'select content from {image_table.path} where id=25').collect()[0]['content']
image_data = base64.standard_b64encode(pet_image).decode("utf-8")

completion = openai_client.chat.completions.create(
    model='shovakeemian-gpt-4o',
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "what's in this image?"},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_data}"},
                },
            ],
        }
    ],
)

In [0]:
#image->image
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
openai_client = w.serving_endpoints.get_open_ai_client()

completion = openai_client.chat.completions.create(
    model='shovakeemian-gpt-4o',
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "generate an image using the image provided. Show the dog jumping for a ball."},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_data}"},
                },
            ],
        }
    ],
)

In [0]:
from openai import OpenAI

DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
  api_key=DATABRICKS_TOKEN,
  base_url="https://e2-demo-field-eng.cloud.databricks.com/serving-endpoints"
)

chat_completion = client.chat.completions.create(
  messages=[
  {
    "role": "system",
    "content": "You are an AI assistant"
  },
  {
    "role": "user",
    "content": "Tell me about Large Language Models"
  }
  ],
  model="shovakeemian-gpt-4o",
  max_tokens=256
)

print(chat_completion.choices[0].message.content)

In [0]:
#image->image
from openai import OpenAI

notebook_token=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
openai_client = OpenAI(
    api_key=notebook_token,
    base_url="https://e2-demo-field-eng.cloud.databricks.com/serving-endpoints"
)

pet_image_bytes=bytes(pet_image)
prompt = """
  Show this dog jumping for a ball.
"""
result = openai_client.images.edit(
    model="shovakeemian-gpt-4o",
    image=("pet_image.png", pet_image_bytes, "image/jpeg"),
    prompt=prompt
)


In [0]:
type(pet_image_bytes)

In [0]:
#image->image
from openai import OpenAI

openai_key=dbutils.secrets.get("shovakeemian-scope", "openai-key")

notebook_token=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
openai_client = OpenAI(api_key=openai_key)

pet_image_bytes=bytes(pet_image)
prompt = """
  Show this dog jumping for a ball.
"""
result = openai_client.images.edit(
    model="gpt-image-1",
    image=("pet_image.png", pet_image_bytes, "image/jpeg"),
    prompt=prompt
)

In [0]:
image_base64 = result.data[0].b64_json
image_bytes = base64.b64decode(image_base64)
Image.open(BytesIO(image_bytes))

In [0]:
#audio->text
#url: https://github.com/rafaelreis-hotmart/Audio-Sample-files/raw/master/sample.mp3 #classical music
#url: https://github.com/Jakobovski/free-spoken-digit-dataset/raw/master/recordings/0_george_0.wav "zero"
#url: https://ia601605.us.archive.org/25/items/MLKDream/MLKDream_64kb.mp3


from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
openai_client = w.serving_endpoints.get_open_ai_client()

audio_url = "https://ia601605.us.archive.org/25/items/MLKDream/MLKDream_64kb.mp3"

completion = openai_client.chat.completions.create(
    model='shovakeemian-gpt-4o',
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Transcribe the following audio file."},
                {
                    "type": "audio_url",
                    "audio_url": {"url": audio_url}, #only allows text or image_url
                },
            ],
        }
    ],
)

# To get the transcription:
print(completion.choices[0].message.content)

In [0]:
from openai import OpenAI

openai_key=dbutils.secrets.get("shovakeemian-scope", "openai-key")

openai_client = OpenAI(api_key=openai_key)

audio_url = "https://ia601605.us.archive.org/25/items/MLKDream/MLKDream_64kb.mp3"
audio_path='/Volumes/ml_shovakeemian/feip/audio_samples/MLKDream_64kb.mp3'

with open(audio_path, "rb") as audio_file:
    transcription = openai_client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file
    )

print(transcription.text)

#testing gemini


In [0]:
Image.open(BytesIO(pet_image))

In [0]:
# OpenAI request
w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

pet_image=spark.sql(f'select content from {image_table.path} where id=25').collect()[0]['content']
image_data = base64.standard_b64encode(pet_image).decode("utf-8")

completion = client.chat.completions.create(
    model="databricks-gemini-3-pro",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "what's in this image?"},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_data}"},
                },
            ],
        }
    ],
)

In [0]:
#image->text
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
openai_client = w.serving_endpoints.get_open_ai_client()

pet_image=spark.sql(f'select content from {image_table.path} where id=25').collect()[0]['content']
image_data = base64.standard_b64encode(pet_image).decode("utf-8")

completion = openai_client.chat.completions.create(
    model='databricks-gemini-2-5-flash',
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "what's in this image?"},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_data}"},
                },
            ],
        }
    ],
)

In [0]:
import base64
import json
import requests
from google.auth import default
from google.auth.transport.requests import Request

PROJECT_IDS=['gcp-dev-field-eng', 'fe-dev-sandbox', 'gcp-sandbox-field-eng']

PROJECT_ID = PROJECT_IDS[2]
LOCATION = "us-central1"
MODEL_ID = "multimodalembedding@001"
VIDEO_PATH = "/Volumes/ml_shovakeemian/feip/video_samples/cat_in_snow.mp4"

ENDPOINT = (
    f"https://{LOCATION}-aiplatform.googleapis.com/v1/"
    f"projects/{PROJECT_ID}/locations/{LOCATION}/publishers/google/"
    f"models/{MODEL_ID}:predict"
)

def get_token():
    creds, _ = default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(Request())
    return creds.token

with open(VIDEO_PATH, "rb") as f:
    video_b64 = base64.b64encode(f.read()).decode("utf-8")

body = {
    "instances": [{"video": {"bytesBase64Encoded": video_b64}}]
}

headers = {
    "Authorization": f"Bearer {get_token()}",
    "Content-Type": "application/json",
}

resp = requests.post(ENDPOINT, headers=headers, data=json.dumps(body))
resp.raise_for_status()
print(json.dumps(resp.json(), indent=2))